# Convert a PyTorch Model to a Spiking Neural Network

**Nuro SDK** — ANN-to-SNN conversion in 3 lines.

Already have a trained PyTorch model? Convert it to an SNN and deploy to neuromorphic hardware — no retraining needed.

In this notebook, you'll:
1. Train a standard PyTorch MLP on MNIST
2. Convert it to an SNN with `nuro.convert_ann()`
3. Run inference as a spiking network
4. See how to deploy it to Loihi, SpiNNaker, or Akida

[![GitHub](https://img.shields.io/badge/GitHub-Vantar--AI%2Fnuro-black)](https://github.com/Vantar-AI/nuro)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue)](https://github.com/Vantar-AI/nuro/blob/main/LICENSE)

## 1. Install

In [ ]:
!pip install -q nuro[gpu] torchvision

## 2. Train a Standard PyTorch MLP

Nothing spiking here — just a normal MLP with ReLU activations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

# Standard PyTorch MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.fc3(x)

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data", train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=256)

print(f"MLP parameters: {sum(p.numel() for p in MLP().parameters()):,}")

In [ ]:
# Train the ANN
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    correct = 0
    total = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = F.cross_entropy(out, target)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == target).sum().item()
        total += target.size(0)
    print(f"Epoch {epoch+1}/3 — Train Accuracy: {100*correct/total:.1f}%")

# Test accuracy
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(1)
        correct += (pred == target).sum().item()
        total += target.size(0)

ann_acc = 100 * correct / total
print(f"\nANN Test Accuracy: {ann_acc:.1f}%")

## 3. Convert to SNN — Three Lines

Nuro's `convert_ann()` walks your model and:
- Maps `nn.Linear` layers to IF neuron populations with dense connections
- Folds `BatchNorm` into the preceding layer's weights
- Absorbs `ReLU` activations into the IF neuron threshold
- Extracts all weights for hardware deployment

The result is a Nuro `Graph` — ready to compile to any backend.

In [ ]:
import nuro

# Convert the trained PyTorch model to an SNN
model.cpu().eval()
snn_graph = nuro.convert_ann(model, input_shape=(784,), num_steps=100)

print(f"Populations: {len(snn_graph.populations)}")
print(f"Connections: {len(snn_graph.connections)}")
for i, pop in enumerate(snn_graph.populations):
    print(f"  Pop {i}: size={pop.size}, dynamics={pop.dynamics}")

## 4. Normalize Weights for Hardware

Before deploying to neuromorphic chips, normalize the weights to match hardware weight ranges. `normalize_weights()` uses percentile-based scaling to avoid outlier issues.

In [ ]:
from nuro.conversion.ann2snn import normalize_weights

# Normalize for hardware deployment
snn_graph = normalize_weights(snn_graph, method="robust", percentile=99.0)

# Check weight ranges
for conn in snn_graph.connections:
    w = conn.params.get("weights")
    if w is not None:
        import numpy as np
        print(f"Connection weights — min: {np.min(w):.3f}, max: {np.max(w):.3f}, shape: {w.shape}")

## 5. Run as SNN on GPU

The converted SNN uses rate coding — pixel intensities become spike rates, and the output neuron with the most spikes wins.

In [ ]:
# Test a few samples
correct = 0
total = 0
num_test_batches = 10  # Test on a subset for speed

for batch_idx, (data, target) in enumerate(test_loader):
    if batch_idx >= num_test_batches:
        break

    data = data.view(-1, 784)
    batch_size = data.shape[0]

    # Rate-code the input
    num_steps = 100
    spike_input = (torch.rand(num_steps, batch_size, 784) < data.unsqueeze(0).abs()).float()

    # Build graph with input
    inp_pop = snn_graph.populations[0]
    out_pop = snn_graph.populations[-1]
    inp = nuro.Input(population=inp_pop, data=spike_input)

    graph = nuro.Graph(
        snn_graph.populations,
        snn_graph.connections,
        inputs=[inp]
    )

    compiled = nuro.compile(graph, target="gpu")
    output = compiled.run(duration=0.1, dt=1e-3, batch_size=batch_size)

    pred = output[out_pop.id].argmax(dim=1)
    correct += (pred == target).sum().item()
    total += batch_size

snn_acc = 100 * correct / total
print(f"SNN Test Accuracy: {snn_acc:.1f}% (on {total} samples)")
print(f"ANN Test Accuracy: {ann_acc:.1f}% (original)")
print(f"Accuracy drop: {ann_acc - snn_acc:.1f}%")

## 6. Deploy to Hardware

The converted SNN compiles to any Nuro backend. Weights are **auto-quantized** for each target.

```python
# Intel Loihi 2 — 8-bit weights, auto-quantized
loihi_model = nuro.compile(graph, target="loihi")
loihi_model.run(duration=0.1)

# SpiNNaker 2 — 16-bit fixed point
s2_model = nuro.compile(graph, target="spinnaker2")

# BrainChip Akida — 1-8 bit, commercially deployed
akida_model = nuro.compile(graph, target="akida")

# All of these run the SAME network with the SAME weights.
# Nuro handles quantization and format conversion automatically.
```

This is the power of ANN-to-SNN conversion: take any trained PyTorch model, convert once, deploy everywhere.

## What's Next?

- [01 — Train an SNN from Scratch](./01_train_mnist_snn.ipynb)
- [03 — Deploy to Neuromorphic Hardware](./03_deploy_to_hardware.ipynb)
- [GitHub](https://github.com/Vantar-AI/nuro) | [Website](https://vantar.xyz) | [Docs](https://github.com/Vantar-AI/nuro/tree/main/docs)